# Agent Roles Library - LangGraph Integration

This notebook demonstrates how to use the Agent Roles Library with **LangGraph**.

## What You'll Learn
- Load role definitions from YAML
- Create LangGraph node builder functions from roles
- Build pre-built graph templates (Sequential, Fan-Out, Debate)
- Execute multi-agent workflows with stateful graphs
- Design custom graph patterns with role-based nodes

## Setup

In [ ]:
import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / "src"))
sys.path.insert(0, str(Path.cwd().parent))

from langchain_core.language_models.fake import FakeListLLM
from langchain_core.messages import HumanMessage

from loader import RoleRegistry
from models import RuntimeContext
from adapters.langgraph_adapter import LangGraphAdapter
from graphs.templates import (
    create_sequential_pipeline_graph,
    create_fan_out_graph,
    create_debate_graph,
    create_reflection_loop_graph,
)

## Step 1: Create an LLM and Context

In [ ]:
# Fake LLM for demonstration
llm = FakeListLLM(
    responses=[
        "Narrative arc: Hero's journey with three acts...",
        "Character profile: Complex protagonist with internal conflict...",
        "Editorial feedback: Strong opening, tighten middle act...",
        "Argument FOR: Innovation drives competitive advantage...",
        "Argument AGAINST: Risk of premature investment...",
        "Verdict: Both positions valid; recommend phased approach...",
    ]
)

# Create runtime context
context = RuntimeContext(
    llm=llm,
    tools=[],
)

print("LLM and context ready")

## Step 2: Index Roles and Load Creative Writing Team

In [ ]:
registry = RoleRegistry()
registry.index()

# Load creative writing roles
narrative_architect = registry.get("narrative_architect")
character_developer = registry.get("character_developer")
developmental_editor = registry.get("developmental_editor")

print("Creative Writing Team:")
for role in [narrative_architect, character_developer, developmental_editor]:
    print(f"  - {role.name}")
    print(f"    Responsibilities: {len(role.responsibilities)} items")
    print(f"    Tools: {role.recommended_tools}")

## Step 3: Sequential Pipeline Graph

Narrative Architect -> Character Developer -> Developmental Editor

In [ ]:
# Create sequential pipeline
pipeline = create_sequential_pipeline_graph(
    roles=[narrative_architect, character_developer, developmental_editor],
    runtime_context=context,
)

print("Sequential Pipeline Graph Created")
print(f"Nodes: {list(pipeline.nodes.keys())}")

In [ ]:
# Execute the pipeline
result = pipeline.invoke({
    "messages": [HumanMessage(content="Write a sci-fi novel outline with a conflicted protagonist")],
    "next": "",
})

print("Pipeline Execution Complete")
print(f"Total messages: {len(result['messages'])}")
for i, msg in enumerate(result["messages"]):
    print(f"\nMessage {i+1} ({msg.type}):")
    print(msg.content[:200] + "...")

## Step 4: Fan-Out Graph

Multiple specialists analyze in parallel, then an aggregator synthesizes.

In [ ]:
# Load risk and audit roles for parallel analysis
operational_risk = registry.get("operational_risk_manager")
compliance = registry.get("compliance_auditor")
it_audit = registry.get("it_auditor")
lead_auditor = registry.get("lead_internal_auditor")

# Create fan-out graph
fan_out = create_fan_out_graph(
    aggregator_role=lead_auditor,
    worker_roles=[operational_risk, compliance, it_audit],
    runtime_context=context,
)

print("Fan-Out Graph Created")
print(f"Nodes: {list(fan_out.nodes.keys())}")

In [ ]:
# Execute fan-out analysis
result = fan_out.invoke({
    "messages": [HumanMessage(content="Assess risks in migrating legacy systems to cloud")],
    "next": "",
})

print("Fan-Out Execution Complete")
print(f"Total messages: {len(result['messages'])}")

## Step 5: Debate / Adversarial Graph

Proposition vs Opposition with Judge evaluation.

Great for: Red-teaming, adversarial validation, critical analysis

In [ ]:
# Load philosophy roles for debate
ethics = registry.get("ethics_advisor")
logic = registry.get("logic_argumentation_analyst")
epistemology = registry.get("epistemology_reviewer")

# Create debate graph
debate = create_debate_graph(
    proposition_role=ethics,
    opposition_role=logic,
    judge_role=epistemology,
    runtime_context=context,
    num_rounds=2,
)

print("Debate Graph Created")
print(f"Nodes: {list(debate.nodes.keys())}")

In [ ]:
# Execute debate
result = debate.invoke({
    "messages": [HumanMessage(content="Should AI systems be granted moral consideration?")],
    "next": "",
    "proposition": "",
    "arguments_for": [],
    "arguments_against": [],
    "rounds": 0,
    "judge_verdict": None,
})

print("Debate Execution Complete")
print(f"Total messages: {len(result['messages'])}")
print(f"Rounds completed: {result.get('rounds', 0)}")

## Step 6: Reflection Loop Graph

Producer generates, Reviewer critiques, Producer revises.

In [ ]:
# Create reflection loop
reflection = create_reflection_loop_graph(
    producer_role=narrative_architect,
    reviewer_role=developmental_editor,
    runtime_context=context,
    max_iterations=2,
)

print("Reflection Loop Graph Created")

In [ ]:
# Execute reflection loop
result = reflection.invoke({
    "messages": [HumanMessage(content="Draft a compelling opening chapter for a mystery novel")],
    "next": "",
    "iteration": 0,
})

print("Reflection Loop Execution Complete")
print(f"Total messages: {len(result['messages'])}")

## Step 7: Visualize Graph Structure

In [ ]:
# Display graph structure for sequential pipeline
from langchain_core.runnables.graph import CurveStyle, NodeColors

try:
    # Try to get Mermaid diagram
    mermaid = pipeline.get_graph().draw_mermaid()
    print("Sequential Pipeline Mermaid Diagram:")
    print(mermaid[:500] + "...")
except Exception as e:
    print(f"Could not generate diagram: {e}")

# Print node structure
print("\nGraph Nodes:")
for node_name in pipeline.nodes:
    print(f"  - {node_name}")

## Summary

This notebook demonstrated:
1. Loading role definitions and creating LangGraph node functions
2. **Sequential Pipeline** — Linear workflow (Narrative -> Character -> Editor)
3. **Fan-Out** — Parallel analysis with aggregation (Risk, Compliance, IT Audit -> Lead Auditor)
4. **Debate** — Adversarial reasoning (Ethics vs Logic -> Epistemology Judge)
5. **Reflection Loop** — Iterative refinement (Producer -> Reviewer -> Producer)
6. Visualizing graph structures

The Agent Roles Library enables you to compose sophisticated multi-agent workflows
by combining framework-agnostic role definitions with LangGraph's powerful stateful graphs.